walking wobbly as if drunk or dizzy.#walk/VERB wobbly/ADJ as/SCONJ if/SCONJ drunk/ADJ or/CCONJ dizzy/ADJ#0.0#0.0
person walks forward then right arm and shoulder stumble to the right hand side two times#person/NOUN walk/VERB forward/ADV then/ADV right/ADJ arm/NOUN and/CCONJ shoulder/NOUN stumble/NOUN to/ADP the/DET right/ADJ hand/NOUN side/NOUN two/NUM time/NOUN#0.0#0.0
a man stumbles walking from side to side.#a/DET man/NOUN stumble/VERB walk/VERB from/ADP side/NOUN to/ADP side/NOUN#0.0#0.0

In [2]:
import numpy as np
data = np.load('joints/003559.npz', allow_pickle=True)
caption = 'walking wobbly as if drunk or dizzy.'
data['log_acceleration_root_motor'].shape

(6, 106, 1)

In [18]:
from train_ga_mdm import *
from tqdm import tqdm
from torch.utils.data import DataLoader, RandomSampler, BatchSampler
from torch.utils.tensorboard import SummaryWriter
from functools import partial
num_steps = 20000
batch_size = 8
pred_len = 0
context_len = 0
fixed_length = context_len + pred_len
# data_path = "joints"
data_path = "test_data"
dataset = NPZDictDataset(data_path, keys=set(DIFFUSE_KEYS + ANSWER_KEYS), pred_len=pred_len, context_len=context_len)
# ---for repeat sampling during debugging---
steps_per_epoch = 10
sampler = RandomSampler(dataset, replacement=True,
            num_samples=batch_size * steps_per_epoch)
batch_sampler = BatchSampler(sampler, batch_size=batch_size, drop_last=True)
# ---for repeat sampling during debugging---
collate_fn = partial(dict_array_collate_fn, pred_len=1)
loader = DataLoader(dataset, batch_sampler=batch_sampler, num_workers=8, collate_fn=collate_fn, pin_memory=True)
num_epochs = num_steps // (batch_size*steps_per_epoch)
save_interval = 5000 // batch_size
model = ga_mdm(fixed_length=fixed_length, pred_len=pred_len)
opt = AdamW(
    # with amp, we don't need to use the mp_trainer's master_params
    model.parameters(),
    lr=1e-3,
    weight_decay=0.0,
    betas=(0.9, 0.999),
)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode='min',
    factor=0.98,
    patience=5,
    min_lr=1e-6,
)
schedule_sampler = UniformSampler(TOTAL_DIFFUSION_STEPS)
step = 0
fail_count = 0
LOG_EVERY_N = int(os.getenv("LOG_EVERY_N", "50"))
max_grad_norm = float(os.getenv("MAX_GRAD_NORM", "1.0"))
logger.info(f"Train steps: {num_steps}")

act_stats = None

gen = iter(loader)
batch, pred = next(gen)
step += 1
t, weights = schedule_sampler.sample(batch['batch_size'])
# TESTING: fixed noise level
# t = torch.ones(batch_size, dtype=torch.int64) * 5
batch, answers, diffuse_shapes = model(batch, pred, t)

2025-12-09 01:43:49,885 | INFO | train_utils | Train steps: 20000


In [39]:
checkpoint = torch.load('lab/checkpoint_step_6240.pt', map_location="cpu")  # or "cuda"
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

ga_mdm(
  (sequence_pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (embed_timestep): TimestepEmbedder(
    (sequence_pos_encoder): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (time_embed): Sequential(
      (0): Linear(in_features=368, out_features=368, bias=True)
      (1): SiLU()
      (2): Linear(in_features=368, out_features=368, bias=True)
    )
  )
  (seqTransDecoder): TransformerDecoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=368, out_features=368, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=368, out_features=368, bias=True)
        )
        (linear1): Linear(in_features=368, out_features=1472, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_featu

In [40]:
batch, pred = next(gen)
t = torch.ones(batch_size, dtype=torch.int64) * 9
captions = batch['text_caption']
with torch.no_grad():
    batch, answers, diffuse_shapes = model(batch, pred, t)

In [41]:
from loss import accumulations
output_mvs = accumulations(batch)
cpu_mvs = {}
# output_mvs = batch
for k, v in output_mvs.items():
    if k in KEY_MAP:
        cpu_mvs[k] = alg.multivector(keys=KEY_MAP[k], values=[v.cpu().numpy() for v in v.values()])
del output_mvs, batch
torch.cuda.empty_cache()

In [44]:
ani_mvs = {}
for k, v in cpu_mvs.items():
    ani_mvs[k] = cpu_mvs[k][..., 0]
root_ani_motors = ani_mvs['root_motor']
full_ani_motors = ani_mvs['chained_body_rotors']
frame = 0
full_motors = full_ani_motors[frame]
root_body_motors = root_ani_motors[frame]
moved_points = [(r)>> e123 for r in full_motors]
len(~root_body_motors[frame] >> moved_points)

21

In [45]:
def bone_pairs(points):
    return [[points[parent], points[child]] for child, parent in kinematic_child_parent]

def xyz_to_point(v):
    x,y,z = v
    return (e0 + x * e1 + y * e2 + z * e3).dual()

def trp(p0, p1):
    """ Translate from p0 to p1. """
    line = p0 & p1
    return ((0.5 * e0) ^ (line | e123)).exp()

# smpl = np.load('motions/model.npz')
# kinematic_child_parent = list(enumerate(smpl['kintree_table'][0][1:22], start=1))
# bone_vertices = smpl['J'][:22]
# bone_points = [xyz_to_point(v) for v in bone_vertices]
root_ani_motors = ani_mvs['root_motor']
full_ani_motors = ani_mvs['chained_body_rotors']

import time

frame = 0
def animation():
    global start
    frame = int(((time.time() - start) * 10 //1 ) % root_ani_motors.shape[1])
    full_motors = full_ani_motors[frame]
    root_body_motors = root_ani_motors[frame]
    moved_points = [e123]+[(r)>> e123 for r in full_motors]
    return [f"{frame}", *bone_pairs(root_body_motors >> moved_points),]
start = time.time()

alg.graph(
    animation,
    grid=True, labels=True, pointRadius=0.2, animate=True
)

In [46]:
captions

['a person jumps once hitting feet together, then lands and has trouble regaining balance.',
 'a man stumbles walking from side to side.',
 'person walks forward then right arm and shoulder stumble to the right hand side two times',
 'a man jumps once and then wobbles a little while moving legs apart.',
 'person walks forward then right arm and shoulder stumble to the right hand side two times',
 'person walks forward then right arm and shoulder stumble to the right hand side two times',
 'a man jumps once and then wobbles a little while moving legs apart.',
 'a person holds their right arm out to the side before letting it drop down.']